# ??nh gi? dots.mocr tr?n 150 m?u ??u

Notebook n?y ??nh gi? 150 file d? ?o?n ??u ti?n theo th? t? s? c?a `image_id` trong `outputs/predictions/dots_mocr/doclaynet_test`. Ch? gi? c?c file c? ?nh t??ng ?ng trong ground truth `test.json`.

C?c metric d?ng l?i t? `src.common.metrics`: Precision, Recall, F1 v? Mean IoU ? ng??ng IoU 0.50; ??nh gi? ch? v? tr? (b? qua nh?n); k?t qu? theo l?p; Precision/Recall/F1 qua c?c ng??ng IoU 0.50?0.95; v? COCO mAP n?u `pycocotools` s?n c?. Timing ???c t?ng h?p n?u file d? ?o?n l?u `inference_time_s`.

> K?t qu? l? ??nh gi? m?u tr?n 150 ?nh, kh?ng ??i di?n cho to?n b? DocLayNet. Ph?p gh?p box s? d?ng greedy IoU nh? implementation metric chung c?a d? ?n.

In [4]:
from pathlib import Path
import json
import sys
from collections import Counter
from html import escape
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Gi? output Unicode ?n ??nh khi ch?y kernel tr?n Windows.
for stream in (sys.stdout, sys.stderr):
    if stream is not None and hasattr(stream, 'reconfigure'):
        try:
            stream.reconfigure(encoding='utf-8', errors='replace')
        except (OSError, ValueError):
            pass

def show_table(headers, rows):
    """Hi?n th? b?ng HTML g?n, kh?ng c?n c?i th?m pandas."""
    head = ''.join(f'<th>{escape(str(value))}</th>' for value in headers)
    body_rows = []
    for row in rows:
        cells = []
        for value in row:
            if isinstance(value, (float, np.floating)):
                text = f'{float(value):.3f}'
            elif isinstance(value, (int, np.integer)):
                text = f'{int(value):,}'
            else:
                text = str(value)
            cells.append(f'<td>{escape(text)}</td>')
        body_rows.append('<tr>' + ''.join(cells) + '</tr>')
    body = ''.join(body_rows)
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

# T?m th? m?c g?c d? notebook ???c m? t? repo root hay th? m?c notebooks.
cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'src/common/metrics.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Kh?ng t?m th?y src/common/metrics.py. H?y m? notebook b?n trong repo DO-AN-261.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.common.metrics import evaluate, evaluate_boxes, evaluate_boxes_multi_iou, coco_map

DATASET = 'doclaynet_test'
METHOD = 'dots_mocr'
N_SAMPLES = 150
IOU_THRESHOLD = 0.50
ANNOTATION_PATH = ROOT / 'datasets' / DATASET / 'processed' / 'annotations' / 'test.json'
PREDICTIONS_DIR = ROOT / 'outputs' / 'predictions' / METHOD / DATASET
print('Repo:', ROOT)
print('Ground truth:', ANNOTATION_PATH)
print('Predictions:', PREDICTIONS_DIR)

ModuleNotFoundError: No module named 'pandas'

## N?p ground truth v? ch?n m?u

Ch?n theo ID s? t?ng d?n, t?i ?a 150 file d? ?o?n h?p l?. C?c prediction kh?ng c? trong ground truth c?a split `test` ???c b? qua.

In [ ]:
raw_coco = json.loads(ANNOTATION_PATH.read_text(encoding='utf-8'))
category_names = {int(c['id']): c['name'] for c in raw_coco['categories']}
gt_all = {int(image['id']): [] for image in raw_coco['images']}
for ann in raw_coco['annotations']:
    gt_all[int(ann['image_id'])].append({'category_id': int(ann['category_id']), 'bbox': ann['bbox']})

prediction_paths = sorted(
    (p for p in PREDICTIONS_DIR.glob('*.json') if p.stem.isdigit()),
    key=lambda p: int(p.stem),
)
predictions = {}
page_times = {}
selected_ids = []
for path in prediction_paths:
    pred = json.loads(path.read_text(encoding='utf-8'))
    image_id = int(pred['image_id'])
    if image_id not in gt_all or image_id in predictions:
        continue
    selected_ids.append(image_id)
    predictions[image_id] = pred.get('detections', [])
    if pred.get('inference_time_s') is not None:
        page_times[image_id] = float(pred['inference_time_s'])
    if len(selected_ids) == N_SAMPLES:
        break

if not selected_ids:
    raise RuntimeError(f'Kh?ng t?m th?y prediction h?p l? trong {PREDICTIONS_DIR}')
ground_truth = {image_id: gt_all[image_id] for image_id in selected_ids}
print(f'S? ?nh ???c ??nh gi?: {len(selected_ids)} / {N_SAMPLES}')
print(f'ID ??u/cu?i: {selected_ids[0]} / {selected_ids[-1]}')
print(f'T?ng v?ng GT: {sum(map(len, ground_truth.values())):,}')
print(f'T?ng v?ng d? ?o?n: {sum(map(len, predictions.values())):,}')
print('ID ?? ch?n:', selected_ids)

## T?nh metric
`strict` y?u c?u kh?p c? class v? bbox. `localization` b? qua class ?? ?o ri?ng kh? n?ng ??nh v?. COCO mAP c?n confidence score v? package `pycocotools`.

In [ ]:
strict = evaluate(ground_truth, predictions, category_names, iou_threshold=IOU_THRESHOLD, require_same_category=True)
localization = evaluate_boxes(ground_truth, predictions, iou_threshold=IOU_THRESHOLD)
multi_iou = evaluate_boxes_multi_iou(ground_truth, predictions)
pred_counts = Counter(d.get('category_id') for items in predictions.values() for d in items)

# COCO mAP ch? t?nh khi ?? score; n?u thi?u dependency th? notebook v?n ho?n th?nh.
detections_with_score = [
    {'image_id': image_id, **d}
    for image_id, items in predictions.items()
    for d in items
    if d.get('score') is not None and d.get('category_id') in category_names
]
all_detection_count = sum(map(len, predictions.values()))
map_result = None
map_note = None
if len(detections_with_score) != all_detection_count:
    map_note = f'B? qua COCO mAP: thi?u score ho?c c? class ngo?i taxonomy ? {all_detection_count - len(detections_with_score)} prediction.'
else:
    image_id_set = set(selected_ids)
    gt_subset = {
        'images': [im for im in raw_coco['images'] if int(im['id']) in image_id_set],
        'annotations': [a for a in raw_coco['annotations'] if int(a['image_id']) in image_id_set],
        'categories': raw_coco['categories'],
    }
    try:
        map_result = coco_map(gt_subset, detections_with_score)
    except ImportError as exc:
        map_note = str(exc)

print('Strict:', strict['overall'])
print('Localization-only:', localization)
print(map_note if map_note else 'COCO mAP ?? t?nh.')

## T?ng h?p v? k?t qu? theo l?p

In [ ]:
s = strict['overall']
l = localization
summary_rows = [
    ['Strict (v? tr? + class)', s['tp'], s['fp'], s['fn'], s['precision'], s['recall'], s['f1'], s['mean_iou_matched']],
    ['Localization-only (b? class)', l['tp'], l['fp'], l['fn'], l['precision'], l['recall'], l['f1'], l['mean_iou_matched']],
]
show_table(['Ph?p ?o', 'TP', 'FP', 'FN', 'Precision', 'Recall', 'F1', 'Mean IoU'], summary_rows)

category_rows = []
for category_id, name in category_names.items():
    m = strict['per_category'].get(category_id, {})
    support = int(m.get('support', 0))
    category_rows.append({
        'Class': name, 'GT': support, 'Pred': pred_counts.get(category_id, 0),
        'TP': int(m.get('tp', 0)), 'FP': int(m.get('fp', 0)), 'FN': int(m.get('fn', support)),
        'Precision': m.get('precision', 0.0), 'Recall': m.get('recall', 0.0), 'F1': m.get('f1', 0.0),
    })
per_class = sorted(category_rows, key=lambda row: (-row['GT'], row['Class']))
show_table(
    ['Class', 'GT', 'Pred', 'TP', 'FP', 'FN', 'Precision', 'Recall', 'F1'],
    [[row[key] for key in ['Class', 'GT', 'Pred', 'TP', 'FP', 'FN', 'Precision', 'Recall', 'F1']] for row in per_class],
)

supported = [row for row in per_class if row['GT'] > 0]
macro = {
    metric: sum(row[metric] for row in supported) / len(supported) if supported else 0.0
    for metric in ['Precision', 'Recall', 'F1']
}
print('Macro (trung b?nh c?c l?p c? GT):')
show_table(['Precision', 'Recall', 'F1'], [[macro['Precision'], macro['Recall'], macro['F1']]])

## Thay ??i theo ng??ng IoU

In [ ]:
iou_rows = multi_iou
show_table(
    ['IoU', 'TP', 'FP', 'FN', 'Precision', 'Recall', 'F1', 'Mean IoU'],
    [[row['iou_threshold'], row['tp'], row['fp'], row['fn'], row['precision'], row['recall'], row['f1'], row['mean_iou_matched']] for row in iou_rows],
)

fig, ax = plt.subplots(figsize=(8, 4.5))
iou_values = [row['iou_threshold'] for row in iou_rows]
for metric, label in [('precision', 'Precision'), ('recall', 'Recall'), ('f1', 'F1')]:
    ax.plot(iou_values, [row[metric] for row in iou_rows], marker='o', label=label)
ax.set(title='dots.mocr ? localization metrics theo ng??ng IoU', xlabel='IoU threshold', ylabel='Score', ylim=(0, 1))
ax.grid(alpha=0.25)
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(8, max(4, 0.38 * len(per_class))))
plot_data = sorted(per_class, key=lambda row: row['F1'])
ax.barh([row['Class'] for row in plot_data], [row['F1'] for row in plot_data], color='#4472C4')
ax.set(title=f'Strict F1 theo class (IoU ? {IOU_THRESHOLD:.2f})', xlabel='F1', xlim=(0, 1))
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.show()

## COCO mAP v? th?i gian suy lu?n
mAP/AP/AR d?ng score c?a prediction. N?u m?i detection c? confidence gi?ng nhau, x?p h?ng g?n nh? kh?ng mang th?ng tin; n?n xem mAP c?ng F1 t?i IoU c? ??nh. Timing ch? hi?n khi c?c file JSON c? `inference_time_s`.

In [ ]:
if map_result is None:
    print('COCO mAP:', map_note or 'kh?ng c? k?t qu?')
else:
    map_overall = map_result['overall']
    map_keys = list(map_overall.keys())
    show_table(map_keys, [[map_overall[key] for key in map_keys]])
    map_classes = sorted(map_result['per_category'].values(), key=lambda item: item['ap'], reverse=True)
    show_table(['Class', 'AP', 'AP50'], [[item['name'], item['ap'], item['ap50']] for item in map_classes])
    scores = [float(d['score']) for d in detections_with_score]
    print(f'S? m?c confidence duy nh?t: {len(set(scores))}; min={min(scores):.3f}, max={max(scores):.3f}')

if page_times:
    values = np.asarray([page_times[i] for i in selected_ids if i in page_times], dtype=float)
    print(f'Timing: {len(values)}/{len(selected_ids)} ?nh c? th?i gian; mean={values.mean():.3f}s, median={np.median(values):.3f}s, p95={np.percentile(values, 95):.3f}s')
else:
    print('Timing: kh?ng c? inference_time_s trong c?c file d? ?o?n ???c ch?n.')

## Ghi ch? khi di?n gi?i
- Precision th?p th??ng ?i c?ng nhi?u false positive; Recall th?p cho th?y c?n nhi?u v?ng ground truth b? b? s?t.
- So s?nh strict v?i localization-only gi?p nh?n bi?t m?t ph?n l?i c? th? ??n t? nh?n class.
- Mean IoU ch? trung b?nh tr?n c?c c?p ?? gh?p; n? kh?ng ph?n ?nh c?c box b? b? s?t.
- ??y l? 150 ?nh ??u theo ID s? trong c?c d? ?o?n hi?n c?. ?? b?o c?o ??i di?n to?n split, ch?y inference v? ??nh gi? tr?n to?n b? ?nh test.